# Violin plot of trace distance, Lorentzian-spectrum dataset
Ported from `03C_plot_violin_tracedist.ipynb`. There are only **5** width points here (vs. 8 gammas before), so — unlike the original plot, which showed Kossakowski only for the 4 most-Markovian gammas and Lindblad only for the 4 least-Markovian ones — this version plots **both ansatzes at all 5 widths**. We don't yet have independent evidence of where the Markovian/non-Markovian crossover falls for the Lorentzian bath, so splitting the display would be an unjustified assumption; once the fits are in, restrict each ansatz to its better half if desired.

Widths are ordered narrow→broad (`wid5..wid1` = 1,2,4,16,64) on the x-axis, mirroring the old plot's high-Q(non-Markovian)-on-the-left convention, since Lorentzian width has no directly analogous published Q-factor formula (that used `Q=8π/γ`, specific to the exponential-bath model).

In [ ]:
import h5py
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import glob
import pickle

In [ ]:
# plot order: narrow (most non-Markovian) -> broad (most Markovian)
wid_list   = ["wid5", "wid4", "wid3", "wid2", "wid1"]
width_val  = {"wid1": 64, "wid2": 16, "wid3": 4, "wid4": 2, "wid5": 1}
x_labels   = [width_val[w] for w in wid_list]
all_states = [f"State{s}" for s in ["0", "1", "X", "Y"]] + [f"Dodeca{i}" for i in range(1, 11)]

In [ ]:
def latest_file(pattern):
    files = sorted(glob.glob(pattern))
    if not files:
        raise FileNotFoundError(f"No file matching: {pattern}")
    print(files[-1])
    return files[-1]

In [ ]:
def read_tracedist(file):
    """Concatenate TraceDist across all 14 states for each width."""
    td_per_width = []
    for w in wid_list:
        with h5py.File(file, "r") as f:
            td_all = []
            for state in all_states:
                td = f[w][state]["TraceDist"][...]
                td_all.extend(td.tolist())
        td_per_width.append(td_all)
    return td_per_width

In [ ]:
kossak_td   = read_tracedist(latest_file("E_KOSSAK_LORENTZIAN_TRACEDIST_ALLSTATES_*.h5"))
lindblad_td = read_tracedist(latest_file("E_LINDBLAD_LORENTZIAN_TRACEDIST_ALLSTATES_*.h5"))

with open("NonMark_Lorentzian.pkl", "rb") as fh:
    NonMark = pickle.load(fh)
# NonMark was computed in wid1..wid5 order; reorder to match wid_list (wid5..wid1)
wid_native_order = ["wid1", "wid2", "wid3", "wid4", "wid5"]
NonMark = [NonMark[wid_native_order.index(w)] for w in wid_list]
NonMark

In [ ]:
color_blind_palette = ["#009E73", "#E69F00", "#CC79A7"]  # Green, Orange, Purple

def set_violin_colors(violin, color, alpha=0.6, zorder=2):
    for body in violin["bodies"]:
        body.set_facecolor(color); body.set_edgecolor("black")
        body.set_alpha(alpha);    body.set_zorder(zorder)
    for part in ["cbars", "cmins", "cmaxes"]:
        violin[part].set_zorder(zorder)
        violin[part].set_color(color)

plt.rcParams.update({"font.size": 14, "axes.labelsize": 16,
                     "xtick.labelsize": 14, "ytick.labelsize": 14,
                     "legend.fontsize": 12, "axes.titlesize": 18})

labels = []
fig, ax1 = plt.subplots(figsize=(9, 6))

positions = np.arange(1, len(wid_list) + 1)
offset = 0.18

# Kossakowski and Lindblad side-by-side (not overlaid) at each width
v1 = ax1.violinplot(kossak_td, positions=positions - offset, widths=0.32)
set_violin_colors(v1, color_blind_palette[0], zorder=2)
labels.append((mpatches.Patch(color=color_blind_palette[0], alpha=0.6),
               "(a) - Kossakowski model"))

v2 = ax1.violinplot(lindblad_td, positions=positions + offset, widths=0.32)
set_violin_colors(v2, color_blind_palette[1], zorder=2)
labels.append((mpatches.Patch(color=color_blind_palette[1], alpha=0.6),
               "(b) - Lindblad ansatz"))

# Non-Markovianity — twin right axis (note: 3 of the 5 widths have exactly
# zero non-Markovianity, which cannot be shown on a log axis and so are
# simply absent from the red curve rather than plotted at zero)
ax2 = ax1.twinx()
ax2.plot(positions, NonMark, marker="+", markersize=10, color="red",
         label=r"(c) - non-Markovianity $\mathcal{N}$")
ax2.set_yscale("log")
ax2.set_ylabel(r"$\mathcal{N}$, non-Markovianity measure", color="red")
ax2.tick_params(axis="y", colors="red")
ax2.spines["right"].set_color("red")
ax2.legend(loc="lower right")

ax1.set_yscale("log")
ax1.set_xticks(positions, x_labels)
ax1.set_xlim(0.4, len(wid_list) + 0.6)
ax1.set_xlabel(r"Lorentzian bath width $\Gamma$")
ax1.set_ylabel(
    r"$T(\rho_{\mathrm{exact}},\rho_{\mathrm{SID}})$, trace distance")
ax1.legend(*zip(*labels), loc="upper left")

plt.tight_layout()
plt.show()
fig.savefig(
    "LORENTZIAN_SID_TRACEDIST_ALLSTATES_LINDBLAD&KOSSAK_vs_nonMarkovianity_color-blind.PDF")